# 🔬 Concept Drift PoC — Comparativa Fases A, B y C
---
**Objetivo**: Visualizar y comparar el rendimiento de las tres estrategias implementadas para combatir el *concept drift* en el dataset CICIDS2017:
- **Fase A**: Modelo Estático (Frozen)
- **Fase B**: Modelo Reactivo (HAT + PHT con reentrenamiento simple)
- **Fase C**: Modelo Proactivo (HAT + PHT + GAN Data Augmentation)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── Cargar Resultados Guardados ──
data_ab = np.load('/mnt/AI-DATA-SSD/alara/CiberIA_O1_A1/CONCEPT_DRIFT/results_ab.npz')
snap_x_a = data_ab['snap_x_a']
snap_f1_a = data_ab['snap_f1_a']
snap_acc_a = data_ab['snap_acc_a']
snap_time_a = data_ab['snap_time_a']

snap_x_b = data_ab['snap_x_b']
snap_f1_b = data_ab['snap_f1_b']
snap_acc_b = data_ab['snap_acc_b']
snap_time_b = data_ab['snap_time_b']

drift_points_ab = data_ab['drift_points']
N_PRETRAIN = data_ab['N_PRETRAIN']

data_c = np.load('/mnt/AI-DATA-SSD/alara/CiberIA_O1_A1/CONCEPT_DRIFT/results_c.npz')
snap_x_c = data_c['snap_x']
snap_f1_c = data_c['snap_f1']
snap_acc_c = data_c['snap_acc']
snap_time_c = data_c['snap_time']
drift_points_c = data_c['drift_points']

print("✅ Datos cargados correctamente.")


In [ ]:
# ════════════════════════════════════════════════════════════════
# 📊  Visualización — Comparativa Fase A vs Fase B vs Fase C
# ════════════════════════════════════════════════════════════════

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 200,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

fig, axes = plt.subplots(3, 1, figsize=(14, 13), sharex=True)
fig.suptitle("Concept Drift PoC — Comparativa A, B y C",
             fontsize=16, fontweight='bold', y=0.98)

# ── Panel 1: F1 Score (Ventana Deslizante) ──
ax1 = axes[0]
ax1.plot(snap_x_a, snap_f1_a, color='#e74c3c', linewidth=1.5,
         alpha=0.85, label='Fase A — Estático (Frozen)')
ax1.plot(snap_x_b, snap_f1_b, color='#2ecc71', linewidth=1.5,
         alpha=0.85, label='Fase B — Reactivo (PHT)')
ax1.plot(snap_x_c, snap_f1_c, color='#8e44ad', linewidth=1.5,
         alpha=0.85, label='Fase C — Reactivo con GAN')

for i, dp in enumerate(drift_points_ab[:20]):
    ax1.axvline(x=dp, color='#f39c12', alpha=0.35, linewidth=1.0,
                linestyle='--', label='Drift detectado' if i == 0 else None)

ax1.axvline(x=N_PRETRAIN, color='#3498db', alpha=0.6, linewidth=1.5,
            linestyle=':', label=f'Fin pre-entrenamiento')

ax1.set_ylabel("F1 Score (ventana deslizante)")
ax1.set_title("Evolución del F1-Score (Ventana = 5,000)")
ax1.legend(loc='lower right', framealpha=0.9)
ax1.set_ylim(-0.05, 1.05)

# ── Panel 2: Accuracy (Acumulativa) ──
ax2 = axes[1]
ax2.plot(snap_x_a, snap_acc_a, color='#e74c3c', linewidth=1.5,
         alpha=0.85, label='Fase A')
ax2.plot(snap_x_b, snap_acc_b, color='#2ecc71', linewidth=1.5,
         alpha=0.85, label='Fase B')
ax2.plot(snap_x_c, snap_acc_c, color='#8e44ad', linewidth=1.5,
         alpha=0.85, label='Fase C')

ax2.axvline(x=N_PRETRAIN, color='#3498db', alpha=0.6, linewidth=1.5,
            linestyle=':', label=f'Fin pre-entrenamiento')

ax2.set_xlabel("Nº de muestras totales (acumulativo)")
ax2.set_ylabel("Accuracy (acumulativa)")
ax2.set_title("Evolución de la Accuracy a lo largo del Stream")
ax2.legend(loc='lower right', framealpha=0.9)
ax2.set_ylim(0.5, 1.02)

# ── Panel 3: Tiempo de CPU ──
ax3 = axes[2]
ax3.plot(snap_x_a, snap_time_a, color='#e74c3c', lw=1.5, alpha=0.85, label='Fase A')
ax3.plot(snap_x_b, snap_time_b, color='#2ecc71', lw=1.5, alpha=0.85, label='Fase B')
ax3.plot(snap_x_c, snap_time_c, color='#8e44ad', lw=1.5, alpha=0.85, label='Fase C')
ax3.axvline(x=N_PRETRAIN, color='#3498db', alpha=0.6, lw=1.5, ls=':', label='Fin pre-train')
ax3.set_xlabel("Muestras")
ax3.set_ylabel("Tiempo de CPU (ms / 500 obj)")
ax3.legend(loc='upper right')

for ax in axes:
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("/mnt/AI-DATA-SSD/alara/CiberIA_O1_A1/CONCEPT_DRIFT/concept_drift_COMPARE_results.png",
            bbox_inches='tight', facecolor='white')
plt.show()
print("\n✅ Gráfica guardada en CONCEPT_DRIFT/concept_drift_COMPARE_results.png")
